In [5]:
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import text_from_rendered
import os
from pathlib import Path

# 프로젝트 루트 디렉토리 설정
# 현재 작업 디렉토리가 notebooks/이면 상위 디렉토리로, 아니면 현재 디렉토리 사용
current_dir = Path.cwd()
if current_dir.name == 'notebooks':
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

# 1. 모델 로드 (최초 실행 시 시간이 걸립니다)
# GPU가 감지되면 자동으로 GPU를 사용합니다.
model_lst = create_model_dict()
converter = PdfConverter(
    artifact_dict=model_lst,
)

# 2. PDF 파일 경로 설정 (프로젝트 루트 기준)
filepath = PROJECT_ROOT / "datasets" / "4795.pdf"
if not filepath.exists():
    raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {filepath}")
print(f"PDF 파일 경로: {filepath}")

# 3. 변환 실행
# converter는 rendered 객체를 반환합니다
rendered = converter(str(filepath))

# rendered 객체에서 마크다운 텍스트 추출
# text_from_rendered는 (텍스트, 형식, 이미지) 튜플을 반환합니다
full_text, output_format, images = text_from_rendered(rendered)
print(f"출력 형식: {output_format}")
print(f"이미지 개수: {len(images) if images else 0}")

# 4. 결과 저장
output_dir = "output_result"
os.makedirs(output_dir, exist_ok=True)

# Markdown 파일 쓰기
with open(os.path.join(output_dir, "output.md"), "w", encoding="utf-8") as f:
    f.write(full_text)

# (선택) 이미지 파일 저장 로직은 images 딕셔너리를 순회하며 저장하면 됩니다.
print(f"변환 완료! {output_dir}/output.md 파일을 확인하세요.")

PDF 파일 경로: /home/wsm/workspace/daily-arxiv-insights-ops/datasets/4795.pdf


Recognizing Text: 100%|██████████| 33/33 [00:14<00:00,  2.21it/s]
Detecting bboxes: 0it [00:00, ?it/s]


출력 형식: md
이미지 개수: 2
변환 완료! output_result/output.md 파일을 확인하세요.


In [ ]:
import os
import json
from pathlib import Path
from tqdm import tqdm

# Marker 라이브러리 임포트
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import text_from_rendered

# 1. 경로 설정
current_dir = Path.cwd()
# notebooks 폴더에서 실행 중이라면 상위로, 아니면 현재 디렉토리
PROJECT_ROOT = current_dir.parent if current_dir.name == 'notebooks' else current_dir

SOURCE_DIR = PROJECT_ROOT / "datasets"
OUTPUT_DIR = SOURCE_DIR / "done"

# 결과 저장 폴더 생성 (없으면 생성)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 2. 모델 로드 (루프 밖에서 한 번만 실행)
print("🚀 모델 로드 중 (RTX 5090 자원 할당)...")
model_lst = create_model_dict()
converter = PdfConverter(artifact_dict=model_lst)

# 3. PDF 파일 목록 가져오기
pdf_files = list(SOURCE_DIR.glob("*.pdf"))
print(f"📂 총 {len(pdf_files)}개의 PDF 파일을 찾았습니다.")

# 4. 반복문 실행
for pdf_path in tqdm(pdf_files, desc="논문 변환 중"):
    try:
        # 출력될 파일 경로 설정 (예: 4795.pdf -> done/4795.md)
        output_file_path = OUTPUT_DIR / f"{pdf_path.stem}.md"
        
        # 이미 변환된 파일이 있다면 건너뛰기 (선택 사항)
        if output_file_path.exists():
            continue

        # 변환 실행
        rendered = converter(str(pdf_path))
        full_text, output_format, images = text_from_rendered(rendered)

        # Markdown 파일 쓰기
        with open(output_file_path, "w", encoding="utf-8") as f:
            f.write(full_text)
            
    except Exception as e:
        print(f"❌ {pdf_path.name} 변환 실패: {e}")

print(f"\n✅ 모든 작업 완료! 결과물 확인: {OUTPUT_DIR}")